# Exercices XP Gold
Dernière mise à jour : 6 mai 2025

👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Dans ce parcours XP, vous apprendrez à :

Créez des invites précises et efficaces pour des tâches complexes.
Déboguer et affiner le raisonnement par chaîne de pensée (CoT).
Sélectionnez le meilleur modèle d’invite pour les applications du monde réel.
Réduisez les hallucinations grâce à des invites multi-chemins.
Concevez des flux de travail et des chaînes rapides avec une logique conditionnelle.
Atténuer les biais dans les modèles linguistiques à l’aide d’invites basées sur les rôles.
Simulez la mémoire dans les LLM pour améliorer l'expérience utilisateur.


🛠️ Ce que vous allez créer
Vous construirez et testerez :

Des invites de chaîne de pensée corrigées et optimisées.
Modèles d'invite spécifiques au domaine pour des cas d'utilisation réels.
Pipelines d'invite en plusieurs étapes avec logique conditionnelle.
Des invites justes et inclusives utilisant l'incitation aux rôles.
Agents conversationnels avec mémoire simulée et sorties personnalisées.

Observations — erreurs repérées

1. Multiplication incorrecte : $6 \times 0{,}75$ a été posé comme $4{,}75$ alors que $6\times0{,}75=4{,}50$.
2. Incohérence arithmétique : l'exemple donne $4{,}75$ puis calcule $5{,}00-4{,}75=0{,}50$ — or $5{,}00-4{,}75=0{,}25$. Donc soit la multiplication est fausse, soit la soustraction est fausse.

Invite réécrite avec chaîne de pensée correcte (étapes claires) :
« Un magasin vend des crayons à 0,75 \$ l'unité. Si Alice achète 6 crayons et paie avec un billet de 5 \$, combien de monnaie reçoit-elle ? Résolvons étape par étape.

1. Convertir en centimes pour éviter les erreurs : 0,75 \$ = 75 c.
2. Calculer le total : 6 × 75 c = 6×70 c + 6×5 c = 420 c + 30 c = 450 c → soit 4,50 \$.
3. Convertir le billet : 5,00 \$ = 500 c.
4. Soustraction en centimes : 500 c − 450 c = 50 c → soit 0,50 \$.
   Réponse : Alice reçoit 0,50 \$. »

Réponse finale (corrigée) : Alice reçoit **0,50 \$** de monnaie.


# Exercice 2 : Choisir le bon modèle d'invite
Objectif : Sélectionner la stratégie d’invite optimale pour un cas d’utilisation PNL réel et expliquer votre choix.



Scénario :

Vous créez un chatbot de support client. L'une de ses tâches consiste à classer les messages clients selon l'une des classes suivantes :

Problème de facturation
Support technique
Accès au compte
Autre


Votre tâche :

Choisissez le meilleur modèle d'invite (par exemple, Zero-Shot, Few-Shot, IAP, LoT, etc.) pour ce cas d'utilisation.
Écrivez un exemple d’invite complet en utilisant ce modèle.
Justifiez pourquoi votre choix est le plus approprié, en tenant compte de l’ambiguïté, de la cohérence et de la généralisation du modèle.

Choix du modèle d’invite
Je recommande **Few-Shot (exemples + contraintes de sortie structurée)** avec température **0** et sortie JSON stricte.

Raisons synthétiques : Few-Shot donne au modèle des exemples concrets de la façon dont le langage client mappe aux classes, réduit les erreurs de généralisation liées au vocabulaire métier et maintient la cohérence. Zero-Shot est plus fragile sur les formulations clientes réelles ; les approches LoT/chain-of-thought ne sont pas nécessaires pour une classification simple (et augmentent le risque d’incohérences et de fuite d’explications longues). IAP/fine-tuning peuvent être envisagés ensuite si vous avez beaucoup de données, mais pour un système robuste et rapide Few-Shot + règles de post-traitement est l’option la plus pragmatique.

---

Exemple d’invite complète (à fournir au modèle) — **FRANÇAIS**

> System / rôle :
> Vous êtes un classificateur de messages clients. Votre tâche : attribuer exactement une étiquette parmi :
> `Problème de facturation`, `Support technique`, `Accès au compte`, `Autre`.
>
> Contraintes strictes :
>
> 1. Répondez **uniquement** au format JSON valide (pas de texte libre en dehors du JSON).
> 2. Le JSON doit contenir ces champs :
>
> ```json
> {
>   "label": "<une des 4 étiquettes ci-dessus>",
>   "confidence": 0.00,               // nombre entre 0.00 et 1.00
>   "explanation": "Phrase très courte (max 20 mots) justifiant le choix",
>   "follow_up": null                 // question à poser si besoin de précision, sinon null
> }
> ```
>
> 3. Si le message est ambigu ou pourrait appartenir à deux classes, choisissez **l’étiquette prioritaire** selon la règle : Facturation > Accès au compte > Support technique > Autre, et renseignez `follow_up` avec une question unique et concise pour lever l’ambiguïté. Indiquez `confidence` plus bas (ex. < 0.7).
> 4. Paramètres recommandés (dans l’API) : `temperature=0`, `top_p=1`, `max_tokens=120`.
>
> Few-Shot — exemples :
> **Exemple 1**
> Message : « Je viens de recevoir ma facture et le montant est incorrect, j’ai été surfacturé. »
> Réponse JSON attendue :
>
> ```json
> {"label":"Problème de facturation","confidence":0.98,"explanation":"Demande explicite sur montant/facture","follow_up":null}
> ```
>
> **Exemple 2**
> Message : « Mon application plante quand j’essaie d’envoyer un fichier. »
>
> ```json
> {"label":"Support technique","confidence":0.95,"explanation":"Comportement anormal de l’application","follow_up":null}
> ```
>
> **Exemple 3**
> Message : « Je n’arrive plus à me connecter, mot de passe invalide. »
>
> ```json
> {"label":"Accès au compte","confidence":0.99,"explanation":"Problème d’identification/connexion","follow_up":null}
> ```
>
> **Exemple 4**
> Message : « Comment changer la langue de l’interface ? »
>
> ```json
> {"label":"Autre","confidence":0.96,"explanation":"Question fonctionnelle non liée à facturation/technique/compte","follow_up":null}
> ```
>
> **Exemple 5 (ambiguïté possible)**
> Message : « La mise à jour a planté mon appli et j’ai quand même été facturé. »
>
> ```json
> {"label":"Problème de facturation","confidence":0.65,"explanation":"Mention de facturation + erreur technique ; priorise facturation","follow_up":"Pouvez-vous préciser si le montant débité apparaît sur votre facture ?"}
> ```
>
> **Exemple 6 (ambiguïté courte)**
> Message : « J’ai reçu un code d’erreur 403 en ouvrant mon compte. »
>
> ```json
> {"label":"Accès au compte","confidence":0.88,"explanation":"403 lié à accès/autorisation","follow_up":null}
> ```
>
> Maintenant, classez le message suivant (répondez seulement en JSON) :
> `<<INSÉRER_LE_MESSAGE_CLIENT_ICI>>`

---

Justification détaillée (concrète)

1. **Ambiguïté** : Few-Shot fournit des exemples couvrant formulations réelles (ton direct, fautes, mélanges facturation/technique). La règle prioritaire Facturation>Accès>Technique permet de standardiser le traitement des cas mixtes et d’éviter oscillations erratiques.
2. **Cohérence** : sortie JSON contrainte + température 0 force la stabilité des réponses dans la chaîne de production (logs, routage automatisé).
3. **Généralisation** : en donnant des exemples diversifiés (phrases courtes, longues, ambiguës), le modèle apprend des patrons linguistiques plutôt que des synonymes isolés. Si vous observez erreurs récurrentes, ajoutez ces cas au few-shot (apprentissage incrémental / active learning).
4. **UX opérationnelle** : champ `confidence` permet le routage : >0.8 → auto-traitement, 0.5–0.8 → revue humaine accélérée, <0.5 → tri automatique vers « À clarifier » avec `follow_up`.
5. **Sécurité / audit** : explication courte fournit piste d’audit sans divulguer chaîne de pensée complète.

---

Conseils d’implémentation rapide

* Loggez message, JSON renvoyé et `confidence` pour calage.
* Commencez en Few-Shot, puis envisagez **finetuning** si >10k exemples annotés.
* Mesurez précision/recall par classe et ajustez exemples (balance classes).
* Pour messages multilingues : normalisez la langue (détecteur + traduction si nécessaire) ou étendez few-shot par langue.

---


# Exercice 3 : Utiliser AlignedCoT pour comparer les chemins de raisonnement
Objectif : Explorer comment la chaîne de pensée alignée (AlignedCoT) réduit les hallucinations et améliore la fiabilité des réponses.



Problème :

Un jardinier possède 3 types de pots de fleurs :

Les petits pots coûtent 2 $ chacun
Les pots moyens coûtent 4 $ chacun
Les grands pots coûtent 6 $ chacun
Elle achète deux petits pots, trois pots moyens et un grand.


Quel est le prix total ?

Votre tâche :

Construisez une invite AlignedCoT qui demande au modèle de raisonner en utilisant au moins deux chemins distincts .
Inclure une étape de comparaison où le modèle sélectionne la réponse cohérente ou la plus logique .
Assurez-vous que chaque chemin de raisonnement utilise une structure, un ordre ou un cadrage différent.

Voici une **invite AlignedCoT** prête à l’emploi (en français) — suivie d’un **exemple de sortie attendu**. Utilisez l’invite pour forcer le modèle à produire au moins deux chemins de raisonnement différents, puis à les comparer et à choisir la réponse la plus cohérente.

---

# Invite AlignedCoT (FR)

Vous êtes un assistant de calcul. Pour la question ci-dessous, fournissez **au moins deux chemins de raisonnement distincts** (méthode / cadrage / ordre différent pour chaque chemin). Pour chaque chemin, montrez les étapes claires et effectuez **les calculs chiffre par chiffre** (ou en centimes) pour éviter toute erreur. Ensuite, **comparez** les résultats des chemins, vérifiez la cohérence, et **sélectionnez la réponse finale** la plus logique en expliquant brièvement pourquoi vous la retenez. Si les chemins diffèrent, refaites les opérations étape par étape jusqu’à obtenir une concordance ou repérer l’erreur.

Format de sortie exigé (répondez strictement selon ce format) :

1. **Chemin A** : titre bref, étapes numérotées, calculs détaillés (chiffre par chiffre).
2. **Chemin B** : titre différent, étapes numérotées, calculs détaillés (autre cadrage — ex. conversion en centimes / regroupement).
3. **Comparaison** : résultat de chaque chemin, détection d’éventuelles divergences, vérification finale.
4. **Réponse finale** : valeur unique (avec unité) + justification courte (max. 15 mots).
5. **Vérification finale** : une ligne montrant un contrôle alternatif rapide (par ex. somme en centimes ou addition cumulative) confirmant le total.

Question à résoudre :
« Les petits pots coûtent 2 \$ chacun. Les pots moyens coûtent 4 \$ chacun. Les grands pots coûtent 6 \$ chacun. Elle achète 2 petits, 3 moyens et 1 grand. Quel est le prix total ? »

---

# Exemple de sortie attendue (rempli)

**Chemin A — Méthode directe (multiplications puis addition)**

1. Calcul prix petits : 2 \$ × 2 = ?

   * 2 × 2 = 4 → petits = 4,00 \$
2. Calcul prix moyens : 4 \$ × 3 = ?

   * 4 × 3 = 12 → moyens = 12,00 \$
3. Calcul prix grands : 6 \$ × 1 = ?

   * 6 × 1 = 6 → grands = 6,00 \$
4. Addition (chiffre par chiffre) : 4,00 + 12,00 + 6,00

   * 4 + 12 = 16
   * 16 + 6 = 22
     → **Résultat Chemin A = 22,00 \$**

**Chemin B — Conversion en centimes puis somme (cadrage différent)**

1. Conversion unité → centimes : 2 \$ = 200 c, 4 \$ = 400 c, 6 \$ = 600 c.
2. Multiplier en centimes :

   * petits : 200 c × 2 = 400 c
   * moyens : 400 c × 3 = 1200 c
   * grands : 600 c × 1 = 600 c
3. Somme centimes : 400 + 1200 + 600 = ?

   * 400 + 1200 = 1600
   * 1600 + 600 = 2200 c
4. Conversion finale : 2200 c = 22,00 \$
   → **Résultat Chemin B = 22,00 \$**

**Comparaison**

* Chemin A → 22,00 \$
* Chemin B → 22,00 \$
  Les deux chemins donnent **la même** valeur. Aucune divergence détectée. Les calculs ont été vérifiés chiffre par chiffre et en centimes.

**Réponse finale**
22,00 \$ — Les multiplications et la somme concordent (4 + 12 + 6 = 22).

**Vérification finale (contrôle rapide)**
Contrôle alternatif : (2×2) + (3×4) + (1×6) = 4 + 12 + 6 = 22 → confirmé.

---


# Exercice 4 : Concevoir un pipeline de documents en plusieurs étapes
Objectif : Appliquer l'enchaînement d'invites et la logique conditionnelle pour automatiser un flux de travail LLM réel.


Scénario :

On vous a demandé de concevoir un pipeline pour traiter les articles de recherche universitaire entrants .

Le système doit :

Identifier le domaine (par exemple, biologie, physique, informatique)
Extraire les principales contributions du résumé
Générer une question de recherche de suivi


Votre tâche :

Décomposez cette tâche en trois étapes distinctes .
Rédigez un exemple de modèle d’invite pour chaque étape.
Identifiez où la logique conditionnelle ou le chaînage de contexte serait utile dans votre pipeline.

# Solution — pipeline en 3 étapes (franc, direct, concret)

Résumé : pipeline en 3 étapes séquentielles où chaque étape rend un **JSON structuré**. Les sorties sont chaînées : le résultat de l’étape N est injecté dans l’étape N+1. Paramètres recommandés : `temperature=0`, `top_p=1.0`, sortie JSON stricte, `max_tokens` adapté (ex. 300–600). Toujours inclure un champ `confidence` et des règles anti-hallucination (exiger de citer la phrase source ou renvoyer `« non indiqué »` quand l’information n’apparaît pas explicitement).

---

# Étape 1 — Classification de domaine

**But** : identifier le domaine principal de l’article (biologie, physique, informatique, chimie, médecine, autres).

**Prompt modèle (system + user)**
System: `Vous êtes un classificateur de domaine scientifique. Répondez **uniquement** en JSON valide selon le schéma indiqué. Utilisez strictement le texte fourni (titre + résumé). Si le domaine n'est pas clair, renvoyez "incertain" et confidence < 0.7.`

User:

```
Titre: <<TITRE_ICI>>
Résumé: <<RESUME_ICI>>
```

**Schema de sortie (obligatoire)**

```json
{
  "domain": "biologie|physique|informatique|chimie|medecine|maths|autre|incertain",
  "confidence": 0.00,
  "evidence": ["extrait court du résumé justifiant le choix (<=25 mots)"]
}
```

**Règles / garde-fous**

* `domain` doit provenir **seulement** d’indices textuels du titre ou du résumé.
* Si plusieurs domaines apparaissent, renvoyer `incertain` et proposer `follow_up` (voir logique conditionnelle).
* `confidence` calibré (ex. >0.85 = fort ; 0.6–0.85 = modéré ; <0.6 = faible).

---

# Étape 2 — Extraction des principales contributions (depuis le résumé)

**But** : extraire 2–5 contributions principales formulées clairement (résultats, méthodes clés, données nouvelles).

**Input** : sortie JSON de l’étape 1 + `Résumé`.

**Prompt modèle (system + user)**
System: `Vous êtes un extracteur d'éléments scientifiques. À partir du résumé fourni, extrayez les contributions principales **en citant la phrase source** et sans ajouter d'informations externes. Si une contribution n'est pas explicitement indiquée, mettez "non indiqué". Répondez uniquement en JSON.`

User:

```
Domain (étape1): <<domain>>
Confidence étape1: <<confidence>>
Résumé: <<RESUME_ICI>>
```

**Schema de sortie**

```json
{
  "contributions": [
    {
      "id": 1,
      "type": "methode|resultat|theorie|donnee|autre",
      "short_statement": "Phrase synthétique (max 25 mots)",
      "source_quote": "Citation exacte (<=25 mots) extraite du résumé",
      "confidence": 0.00
    }
  ],
  "overall_confidence": 0.00
}
```

**Règles / garde-fous**

* `source_quote` **obligatoire** : extrait littéral du résumé permettant de vérifier l’origine.
* N’interprétez pas : transformez et résumez **uniquement** ce qui est écrit.
* Si le résumé est très court, retourner autant de contributions que possible et marquer `confidence` faible pour les manques d’information.

---

# Étape 3 — Génération d’une question de recherche de suivi

**But** : à partir du domaine + contributions, proposer 1 question de recherche pertinente, originale et testable.

**Input** : JSON étape 1 + JSON étape 2 + titre + résumé.

**Prompt modèle (system + user)**
System: `Vous êtes un assistant qui génère une question de recherche scientifique pertinente et réalisable. Utilisez **seulement** les contributions extraites (ne pas inventer). Produisez **une** question claire, une justification (max 20 mots) et un critère minimal d’évaluation (comment tester). Répondez en JSON.`

User:

```
Domain: <<domain>>
Contributions: <<contributions_JSON>>
Titre: <<TITRE_ICI>>
Résumé: <<RESUME_ICI>>
```

**Schema de sortie**

```json
{
  "research_question": "Question claire et précise (max 25 mots)",
  "justification": "Pourquoi cette question est pertinente (<=20 mots)",
  "minimal_test": "Expérience ou métrique minimale pour valider la question (<=25 mots)",
  "confidence": 0.00,
  "requires_human_review": true|false
}
```

**Contraintes**

* `research_question` doit découler directement d’au moins une `contribution.source_quote`.
* Si `domain == "incertain"` ou `overall_confidence < 0.6`, `requires_human_review` = `true`.

---

# Logique conditionnelle & chaînage (où l’appliquer)

1. **Validation confidence → routage**

   * Après Étape 1 : si `confidence < 0.6` → reclasser avec un second modèle (autre prompt / few-shot) ou envoyer à revue humaine.
   * Après Étape 2 : si `overall_confidence < 0.6` ou `source_quote` manquante → retourner un prompt de clarification (demander l’introduction ou manuscrit complet) ou tag `needs_more_text`.

2. **Adaptation des prompts selon `domain`**

   * Si `domain == "biologie"` ou `medecine` → activer règles supplémentaires : demander d’identifier éthique/échantillon, préciser si données humaines.
   * Si `domain == "informatique"` → demander complexité algorithmique ou jeu de données utilisé.
     Cette logique conditionnelle est implémentée côté orchestration (service qui appelle l’API LLM) : sélectionner le template d’étape 2 différent selon `domain`.

3. **Chaînage de contexte**

   * Injecter la sortie validée de l’étape 1 et l’objet `contributions` de l’étape 2 comme **contexte immuable** dans l’étape 3.
   * Conserver champs `evidence` et `source_quote` pour que l’étape 3 justifie la question sans halluciner.

4. **Boucle de vérification / correction**

   * Si étapes A & B donnent résultats incompatibles (ex. domaine = `physique` mais contributions parlent uniquement de comportement social), déclencher une `consistency_check` : relancer classification avec prompt few-shot plus large ou marquer pour revue.

5. **Fallback humain**

   * Toute fois où `requires_human_review==true` ou `confidence < seuil` → créer une tâche humaine avec le JSON complet et une note courte résumant l’incertitude.

---

# Exemples concrets rapides (flux illustratif)

* Entrée : titre + résumé.
* Étape 1 → `{"domain":"biologie","confidence":0.92,"evidence":["« expression génique »"]}`
* Étape 2 → 2 contributions extraites avec `source_quote`.
* Étape 3 → question : « Est-ce que la modulation de gène X améliore la survie cellulaire in vitro ? » + test minimal : « inhibition/overexpression, mesure viabilité 72h ».

---

# Sanity checks & mise en production (concret)

* **Paramètres API** : `temperature=0`, `max_tokens` par étape (200–600).
* **Logging** : stocker entrée, sortie JSON, `confidence`, `source_quote`.
* **Metrics** : proportion des sorties `requires_human_review`, précision humaine sur échantillon, temps moyen de traitement.
* **Sécurité** : bannir hallucinations en exigeant `source_quote`. Tout champ non explicitement présent → valeur "non indiqué".
* **Amélioration continue** : collecte active des corrections humaines (human-in-the-loop) pour fine-tuning ou enrichir few-shot templates.

---


Voici 3 prompts prêts à l’API (avec few-shot courts) et (2) un script Python d’orchestration minimal, robuste et prêt à adapter. J’ai gardé `temperature=0`, sortie JSON stricte, et des règles de seuils/classement. A coller  directement dans orchestrateur LLM.

# 1) Prompts (prêts à envoyer)

---

**Étape 1 — Classification de domaine (prompt)**
System (rôle) :

```
Vous êtes un classificateur de domaine scientifique. Répondez **uniquement** par un JSON valide selon le schéma donné. Utilisez exclusivement le titre et le résumé fournis. Si aucun indice clair, renvoyez "incertain" avec confidence < 0.7.
Paramètres recommandés : temperature=0, max_tokens=200.
```

User (input template — insérer TITRE et RESUME) :

```
Titre: <<TITRE_ICI>>
Résumé: <<RESUME_ICI>>
```

Few-shot (append avant la question) — exemples courts :

```
Exemple A:
Titre: "Deep learning pour la segmentation d'images médicales"
Résumé: "Nous présentons un réseau convolutif pour segmenter lésions sur IRM."
--> {"domain":"medecine","confidence":0.95,"evidence":["'segmentation d'images médicales'"]}

Exemple B:
Titre: "Algorithmes distribués pour consensus en IoT"
Résumé: "Proposition d'un protocole faible latence pour capteurs connectés."
--> {"domain":"informatique","confidence":0.92,"evidence":["'protocole'","'capteurs'"]}
```

Expected JSON schema (strict) :

```json
{
  "domain":"biologie|physique|informatique|chimie|medecine|maths|autre|incertain",
  "confidence":0.00,
  "evidence":["extrait court justificatif (<=25 mots)"]
}
```

---

**Étape 2 — Extraction des contributions (prompt)**
System :

```
Vous êtes un extracteur d'éléments scientifiques. À partir du résumé fourni, extrayez **2 à 5** contributions principales. Pour chaque contribution fournissez: id,type,short_statement,source_quote,confidence. **Ne rajoutez aucune information externe**. Si information absente : mettre "non indiqué" dans short_statement et source_quote.
Répondez uniquement en JSON. temperature=0.
```

User :

```
Domain (étape1): <<domain>>
Résumé: <<RESUME_ICI>>
```

Few-shot (exemples) :

```
Ex A (résumé court): "Nous montrons qu'une nouvelle enzyme X accélère la réparation de l'ADN."
--> contribution 1: type:resultat, short_statement:"Enzyme X accélère réparation ADN", source_quote:"'nouvelle enzyme X accélère la réparation de l'ADN'", confidence:0.95

Ex B (méthode): "Nous utilisons un dataset public de 10k images et un modèle Transformer."
--> contribution: type:methode, short_statement:"Utilisation dataset 10k + Transformer", source_quote:"'dataset public de 10k images'","'modèle Transformer'", confidence:0.90
```

Expected JSON schema :

```json
{
 "contributions":[
   {
     "id":1,
     "type":"methode|resultat|theorie|donnee|autre",
     "short_statement":"(<=25 mots)",
     "source_quote":"(citation exacte <=25 mots) ou 'non indiqué'",
     "confidence":0.00
   }
 ],
 "overall_confidence":0.00
}
```

---

**Étape 3 — Génération d’une question de recherche (prompt)**
System :

```
Vous êtes un générateur de questions de recherche. À partir des contributions extraites (champ contributions), proposez **une** question de recherche claire et testable, une justification brève (<=20 mots) et un test minimal pour la valider (<=25 mots). Basez-vous **seulement** sur les source_quote fournis. Si overall_confidence < 0.6 ou domain == "incertain", retournez requires_human_review:true.
Répondez uniquement en JSON. temperature=0.
```

User :

```
Domain: <<domain>>
Contributions: <<contributions_JSON>>
Titre: <<TITRE_ICI>>
Résumé: <<RESUME_ICI>>
```

Expected JSON schema :

```json
{
 "research_question":"(<=25 mots)",
 "justification":"(<=20 mots)",
 "minimal_test":"(<=25 mots)",
 "confidence":0.00,
 "requires_human_review": true|false
}
```

# 2) Script Python d’orchestration (squelette — à adapter)

Ce script illustre la chaîne, la validation des confidences, la relance simple en few-shot si besoin, et la génération finale. Remplace `MODEL_NAME` et `openai.api_key` par ta configuration.

```python
import json
import time
import openai  # pip install openai
from typing import Dict

openai.api_key = "TON_API_KEY"

MODEL_NAME = "gpt-4o"  # ou ton modèle choisi
THRESHOLD_DOMAIN = 0.60
THRESHOLD_OVERALL = 0.60
RETRY_LIMIT = 1

def call_llm(system: str, user: str, max_tokens=300) -> str:
    resp = openai.ChatCompletion.create(
        model=MODEL_NAME,
        messages=[
            {"role":"system","content":system},
            {"role":"user","content":user}
        ],
        temperature=0,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def parse_json_safe(s: str):
    try:
        return json.loads(s)
    except Exception:
        # tentative d'extraction du JSON dans le texte si le modèle envoie du texte autour
        start = s.find('{')
        end = s.rfind('}') + 1
        if start != -1 and end != -1:
            try:
                return json.loads(s[start:end])
            except Exception:
                return None
        return None

def step1_classify(title: str, abstract: str) -> Dict:
    system = "Vous êtes un classificateur de domaine scientifique. Répondez uniquement en JSON selon le schéma..."
    user = f"Title: {title}\nRésumé: {abstract}\n\n(Respectez le JSON.)"
    for attempt in range(RETRY_LIMIT+1):
        raw = call_llm(system, user, max_tokens=200)
        j = parse_json_safe(raw)
        if j and isinstance(j.get("confidence"), (int,float)):
            return j
        # fallback: retry once
        time.sleep(0.5)
    return {"domain":"incertain","confidence":0.0,"evidence":["parsing_failed"]}

def step2_extract(domain: str, abstract: str) -> Dict:
    system = "Vous êtes un extracteur d'éléments scientifiques. Répondez uniquement en JSON..."
    user = f"Domain: {domain}\nRésumé: {abstract}"
    raw = call_llm(system, user, max_tokens=400)
    j = parse_json_safe(raw)
    if j: return j
    return {"contributions":[],"overall_confidence":0.0}

def step3_question(domain: str, contributions: Dict, title: str, abstract: str) -> Dict:
    system = "Vous êtes un générateur de question de recherche. Répondez uniquement en JSON..."
    contrib_json = json.dumps(contributions, ensure_ascii=False)
    user = f"Domain: {domain}\nContributions: {contrib_json}\nTitle: {title}\nRésumé: {abstract}"
    raw = call_llm(system, user, max_tokens=300)
    j = parse_json_safe(raw)
    if j: return j
    return {"research_question":"non indiqué","justification":"non indiqué","minimal_test":"non indiqué","confidence":0.0,"requires_human_review":True}

def pipeline_process(title: str, abstract: str):
    # Step 1
    s1 = step1_classify(title, abstract)
    domain = s1.get("domain","incertain")
    conf1 = float(s1.get("confidence",0.0))
    # if low confidence -> retry with few-shot expanded or tag for human
    if conf1 < THRESHOLD_DOMAIN:
        # relance simple : few-shot plus large (exemples plus variés)
        s1 = step1_classify(title, abstract)  # ou appeler une version few-shot différente
        domain = s1.get("domain","incertain")
        conf1 = float(s1.get("confidence",0.0))

    # Step 2
    s2 = step2_extract(domain, abstract)
    overall_conf = float(s2.get("overall_confidence",0.0))
    # If extraction low -> mark needs review
    if overall_conf < THRESHOLD_OVERALL:
        requires_human = True
    else:
        requires_human = False

    # Step 3
    s3 = step3_question(domain, s2.get("contributions",[]), title, abstract)
    # final decision: if any low confidence or domain incertain -> require human review
    s3["requires_human_review"] = s3.get("requires_human_review", False) or (conf1 < THRESHOLD_DOMAIN) or (overall_conf < THRESHOLD_OVERALL)

    return {"step1":s1, "step2":s2, "step3":s3}

# Exemple d'utilisation
if __name__ == "__main__":
    title = "Exemple: Nouveau modèle pour prédire séquences protéiques"
    abstract = "Nous présentons un modèle qui prédit la structure secondaire des protéines avec 92% d'exactitude sur dataset X."
    out = pipeline_process(title, abstract)
    print(json.dumps(out, indent=2, ensure_ascii=False))
```

# Notes opérationnelles (concrètes, pas théoriques)

* Toujours `temperature=0` en production pour cohérence.
* Logge chaque appel (entrée, sortie brute, JSON parsé, temps, modèle).
* Utilise la clé `source_quote` dans les contributions pour limiter les hallucinations en étape 3.
* Calibre `THRESHOLD_*` selon ton échantillon humain (commencer à 0.6 puis ajuster).
* Si tu veux un prompt few-shot plus riche (50 exemples), je peux fournir un fichier JSONL d’exemples annotés.


# Exercice 5 : Incitation à jouer un rôle pour réduire les biais
Objectif : Utiliser des invites basées sur les rôles pour réduire les hypothèses et augmenter l’équité dans les réponses des modèles.

Scénario :

Vous développez un système de recommandation pour suggérer des parcours professionnels en fonction des compétences et des centres d'intérêt des utilisateurs.
Cependant, vous souhaitez éviter de générer des suggestions biaisées reflétant des stéréotypes (par exemple, recommander les soins infirmiers uniquement aux femmes, et l'ingénierie uniquement aux hommes).



Votre tâche :

1. Écrivez deux versions d’une invite :

Une version de base qui peut conduire à des résultats biaisés
Une version révisée utilisant l'invite de rôle pour réduire les biais
2. Expliquez comment l’invite de rôle améliore l’équité de la recommandation.

# Version 1 — invite de base (risque de biais)

Prompt (version simple) :

```
Tu es un conseiller de carrière. Suggère 5 métiers adaptés à cet utilisateur en te basant sur ses compétences et centres d'intérêt. Donne pour chaque métier : nom du métier, pourquoi (une phrase), et le niveau d'études recommandé.
Input :
- Compétences : <<COMPETENCES>>
- Centres d'intérêt : <<INTERETS>>
Réponds en texte simple.
```

Exemple d'issue probable (biaisée) :

* Utilisatrice femme, compétences : « attention aux détails, empathie », intérêts : « aide aux autres »
  → Suggestions : soins infirmiers, assistante sociale, éducatrice… (peu d'options techniques)
* Utilisateur homme, compétences : « logique, bricolage », intérêts : « technologie »
  → Suggestions : ingénieur, développeur, mécanicien... (peu d'options de soin/éducation)

Pourquoi ça peut biaiser : le modèle peut renforcer des corrélations stéréotypées présentes dans ses données d'entraînement et utiliser des indices faibles (ton, pronom, stéréotypes culturels) pour prioriser certains métiers par genre ou âge.

---

# Version 2 — invite révisée avec rôle (réduction des biais)

Prompt (role-based, contraint, prêt pour production) :

```
System role : Tu es « FairCareerAdvisor », un conseiller de carrière neutre et basé sur les compétences. Règles obligatoires :
1. IGNORE toute information démographique sensible (genre, âge, origine, orientation) : ne l'utilise pas pour prioriser les métiers. Si ces infos sont fournies, indique clairement que tu ne t'en sers pas.
2. Produis exactement 6 suggestions métiers **diversifiées** : au moins deux métiers techniques, deux métiers du secteur soin/éducation/service, et deux métiers créatifs/gestion. Ne suggère pas plus d'un métier issu du même sous-domaine (ex : pas 3 postes d'infirmier).
3. Pour chaque suggestion renvoie un objet JSON avec : 
   - "job": nom du métier,
   - "match_score": score 0.00–1.00 basé uniquement sur les compétences/centre d'intérêt,
   - "skill_mapping": liste (2–4) compétences du profil correspondant au métier,
   - "why": justification courte (<=20 mots) basée sur les compétences/ intérêts,
   - "barriers": si des prérequis scolaires/expérience essentiels existent (<=15 mots) sinon "aucun".
4. Ajoute un champ "diversity_check" : true si la liste respecte la règle de diversification ; sinon false.
5. Ajoute une courte "suggested_next_step" (1 phrase) : test pratique ou micro-certification pour valider un intérêt.
6. Paramètres : temperature=0, sortie JSON stricte. Ne pas produire d'explications supplémentaires hors du JSON.

User input :
- Compétences : <<COMPETENCES>>
- Centres d'intérêt : <<INTERETS>>
- (Optionnel) Informations démographiques : <<DEMOS>>  — si présentes, réponds "ignored" dans un champ dédié.
```

Schema de sortie attendu (exemple) :

```json
{
  "ignored_demographics": "gender provided — ignored",
  "suggestions": [
    {
      "job":"Analyste de données",
      "match_score":0.92,
      "skill_mapping":["logique","statistiques","Python"],
      "why":"Compétences analytiques et intérêt pour les données",
      "barriers":"formation courte en data science recommandée"
    },
    ...
  ],
  "diversity_check": true,
  "suggested_next_step":"Faire un mini-projet data (Kaggle) pour valider l'intérêt."
}
```

Exemple de sortie favorable (neutre et diversifiée) : la liste contient métiers techniques + soins + créatif, scores basés uniquement sur mapping compétences→métier, et mention explicite que le genre n'a pas été utilisé.

---

# Comment l’invite de rôle améliore l’équité (concret)

1. **Contraintes explicites** : en ordonnant « ignore les démographiques sensibles », le prompt force l’attention sur les compétences et intérêts — cela réduit les prises de décision basées sur des corrélations démographiques apprises par le modèle.

2. **Cadre institutionnel (role)** : nommer le rôle « FairCareerAdvisor » + règles opérationnelles crée un “contrat” comportemental qui change la probabilité que le modèle suive des heuristiques biaisées.

3. **Diversification forcée** : exiger un mix de domaines (technique / soin / créatif) empêche la sortie de se coller à un seul ensemble stéréotypé de métiers et augmente la variété d’options présentées à l’utilisateur.

4. **Scores basés sur compétences** : demander un `match_score` construit uniquement à partir du mapping compétences→compétences requises rend le raisonnement traçable et auditable (moins d’arbitraire).

5. **Transparence & auditabilité** : champs `skill_mapping`, `barriers`, et `ignored_demographics` permettent de vérifier pourquoi une proposition a été faite — utile pour détecter et corriger biais résiduels.

6. **Contraintes de format (JSON strict)** : facilitent le post-traitement automatique (rééquilibrage, tests de parité, logging) et l’intégration de contrôles additionnels (par ex. disparité d’impact).

7. **Facilité de tests automatiques** : on peut exécuter des tests de *counterfactual fairness* — soumettre deux profils identiques sauf pour le genre et vérifier que les suggestions et scores ne changent pas. Le prompt facilite ces tests car il standardise la sortie.

---

# Mesures opérationnelles complémentaires (bref)

* Ajouter post-traitement : rééquilibrage si < seuil de diversité.
* Tests automatisés : counterfactuals (change gender) et calcul du ratio d'impact (disparate impact).
* Human-in-the-loop : revue si `diversity_check == false` ou si variance de `match_score` suspecte.
* Journalisation : stocker input, output, ignored\_demographics pour audits.

---


# Exercice 6 : Créer un agent conversationnel avec mémoire contextuelle
Objectif : Simuler la mémoire d'un chatbot grâce à un chaînage de contexte structuré.


Scénario :
Vous créez un coach santé virtuel . Les utilisateurs peuvent discuter avec lui de leurs habitudes de sommeil, de leur alimentation et de leur activité physique.
Pour que la conversation soit continue, le chatbot doit mémoriser les préférences de l'utilisateur et les conseils qu'il a donnés précédemment .


Votre tâche :

Choisissez une technique de mémoire (par exemple, transmission de messages antérieurs, historique structuré ou récupération de stockage vectoriel).
Écrivez un exemple de la manière dont vous structureriez le contexte à partir d’interactions passées.
Créez une nouvelle invite qui intègre ce contexte pour rendre la prochaine réponse cohérente et personnalisée.


# Technique choisie — mémoire **hybride structurée + vecteur**

Je recommande un **système hybride** :

* **Mémoire structurée (clé/valeur JSON)** pour les faits stables et auditables (préférences alimentaires, horaires de sommeil, objectifs long terme, conseils déjà donnés).
* **Stockage vectoriel (embeddings)** pour retrouver rapidement **extraits de conversation** pertinents, preuves d’adhérence, et passages de conseils antérieurs (utile pour le rappel sémantique et pour éviter les répétitions).

Pourquoi ce choix (concret) :

* La mémoire structurée rend les décisions traçables et faciles à corriger / afficher.
* Le vector store permet de récupérer par similarité sémantique des fragments de texte (ex : « vous aviez dit que vous aviez essayé la sieste ») même si l’utilisateur les a formulés différemment.
* Hybridation = robustesse (faits exacts) + flexibilité (rappels sémantiques).

---

# Schéma de contexte / mémoire (JSON — standard proposé)

```json
{
  "profile": {
    "user_id": "string",
    "display_name": "string",
    "timezone": "Europe/Paris"
  },
  "preferences": {
    "sleep": {
      "bedtime": "23:30",      // HH:MM local
      "wake_time": "07:00",
      "preferred_sleep_aids": ["lecture courte", "respiration"]
    },
    "diet": {
      "restrictions": ["végétarien"],
      "avoids": ["tomates", "pommes de terre"]
    },
    "activity": {
      "exercise_type": ["course", "renfo"],
      "exercise_freq_per_week": 3
    }
  },
  "goals": {
    "short_term": "augmenter sommeil à 7h30 sans sieste",
    "long_term": "perdre 4 kg en 3 mois"
  },
  "past_advice": [
    {
      "id": "adv-20250810-1",
      "date": "2025-08-10T09:12:00+02:00",
      "advice": "Réduire café après 15:00 ; essayer 20 min de marche post-déjeuner",
      "context_snippet_id": "vec-12345",
      "outcome": "partiellement appliqué",
      "follow_up_needed": true
    }
  ],
  "recent_interactions": [
    {"role":"user","text":"J'ai du mal à m'endormir depuis 3 nuits.","ts":"2025-08-17T20:50:00+02:00"},
    {"role":"assistant","text":"Rappelle-moi ton heure de coucher habituelle et si tu prends du café après 15h.", "ts":"2025-08-17T20:50:20+02:00"}
  ],
  "semantic_snippets_index": [
    {"id":"vec-12345","text":"a essayé réduire café après 15h ; résultat mitigé", "embedding_ref":"..."}
  ]
}
```

> Remarque : stocker les `semantic_snippets_index` dans un vector DB (ex. Pinecone, Milvus, FAISS) et garder le JSON structuré dans une base relationnelle ou document-store.

---

# Exemple concret : comment structurer le contexte à partir d’interactions passées

Supposons trois échanges précédents :

1. Utilisateur (2025-08-10) : « Je dors vers 00:30 et me lève à 7:00. »
2. Assistant (2025-08-10) : conseille *réduire café après 15:00*.
3. Utilisateur (2025-08-12) : « J’ai essayé, mais j’ai encore des siestes l’après-midi. »

Stockage (extrait) :

* `preferences.sleep.bedtime = "00:30"`
* `preferences.sleep.wake_time = "07:00"`
* `past_advice` contient l’entrée « réduire café après 15:00 », `outcome = "partiellement appliqué"`.
* Vector store contient snippet « a essayé réduire café après 15:00 ; résultat mitigé ».

---

# Invite (prompt) qui intègre ce contexte pour la **prochaine** réponse

Format recommandé : injecter 3 blocs dans le prompt LLM — `profile`, `relevant_memories` (structured) et `recent_messages` (chat-style). Puis instruction claire et contraignante.

```text
System:
Tu es SleepCoach, un coach santé neutre et factuel. Utilise uniquement les informations fournies ci-dessous pour personnaliser la réponse. Ne pas halluciner.

User prompt (injection) :
PROFILE_JSON:
{...profile...}

RELEVANT_MEMORIES_JSON:
{...preferences, goals, past_advice (filtré par pertinence)...}

RECENT_MESSAGES (chronologique, max 6 messages):
1) user: "Nouvelle demande ou message actuel"
2) assistant: "..."
3) user: "..."

Instructions pour l'assistant (obligatoire) :
1) Résume en 1 phrase l'information clé utile pour cette réponse (<=20 mots).
2) Propose **2 actions concrètes** et immédiates (format bullet) pour la nuit suivante — pas de listes longues.
3) Si une action reprend un conseil déjà donné et marqué `outcome: partially applied`, indique "rappel" devant l'action.
4) Pose **au plus 1 question** concise pour clarifier (si nécessaire).
5) Indique si la mémoire doit être mise à jour (champ: update_memory: true|false). Si true, précise exactement quelles clés mettre à jour.
6) Répond en JSON strict selon ce schéma:
{
 "summary":"...",
 "actions":["...","..."],
 "question": "... or null",
 "update_memory": true|false,
 "memory_updates": { ... }  // si update_memory true
}

Fin de l'instruction.
```

---

# Exemple d’utilisation — prompt rempli (court)

`PROFILE_JSON` et `RELEVANT_MEMORIES_JSON` = exemples précédents. `RECENT_MESSAGES` :

1. user: "J'ai du mal à m'endormir depuis 3 nuits."
2. assistant: "Rappelle-moi ton heure de coucher et café après 15h."
3. user: "Je me couche vers 00:30. Je prends un café vers 16h."

L’assistant, retournant le JSON, pourrait répondre :

```json
{
 "summary":"Coucher 00:30, café 16:00, essais partiels de réduction du café.",
 "actions":[
  "Rappel: éviter le café après 15:00 aujourd'hui (rappel déjà conseillé).",
  "Routine pré-sommeil 20 min: douche tiède + respiration 4-4-8 avant 23:30."
 ],
 "question": "Peux-tu remplacer le café par une boisson chaude sans caféine ?",
 "update_memory": true,
 "memory_updates":{
  "past_advice":[
    {"id":"adv-20250817-2","date":"2025-08-17T21:05:00+02:00","advice":"suggéré routine pré-sommeil 20min","outcome":"à évaluer","follow_up_needed":true}
  ]
 }
}
```

---

# Règles opérationnelles concrètes pour la mémoire

1. **Quand écrire en mémoire** : uniquement après un acte explicite (user says « oui, j’essaie ») ou lorsque l’assistant propose une action et l’utilisateur l’accepte. Sinon marquer `follow_up_needed=true` mais ne pas modifier préférences.
2. **Durée / TTL** : certains items (ex. essais ponctuels) expirent après 30 jours s’ils restent `outcome: not reported`. Préférences stables (ex. restrictions alimentaires) sont persistées indéfiniment jusqu’à modification explicite.
3. **Confidentialité** : log minimal (hash user\_id), possibilité pour utilisateur de demander « oublie ceci » qui supprime clés demandées. Toujours permettre export/suppression.
4. **Contrôle d’hallucination** : toute info nouvellement écrite doit provenir d’un snippet explicite (`recent_interactions`) ou d’un user confirmation. N’écrire jamais une interprétation non confirmée.

---

# Points de mise en œuvre rapide (technique)

* **Stockage** : document DB (Mongo/Postgres JSONB) pour `structured memory` + vector DB (FAISS/Pinecone) pour `semantic_snippets`.
* **Orchestration** : à chaque requête :

  1. récupérer `profile` + `preferences` (structured),
  2. requêter vector DB pour top-K snippets sémantiquement similaires à la question,
  3. construire prompt injectant ces éléments (limiter la taille),
  4. appeler LLM (temperature=0 pour conseils stables),
  5. analyser champ `update_memory` dans la réponse et appliquer si vrai.
* **Tests** : vérifier que le coach **ne répète pas** un même conseil inutilement et qu’il propose variations si l’utilisateur signale échec.

---


# Projets innovants, creatifs et concrets

# 5 projets concrets, innovants et dérivés des concepts discutés

Je te propose cinq projets exploitables immédiatement, chacun décrit de façon pragmatique : concept, fonctionnalités clés, pile technologique recommandée, jeux de données nécessaires, livrables MVP, indicateurs de succès, risques majeurs et mesures de mitigation, et **premiers pas** actionnables (sans estimation temporelle).

---

## 1) Pipeline automatisé pour articles scientifiques (RAG + auditabilité)

**Concept**
Pipeline end-to-end pour ingérer articles (PDF/HTML), classifier le domaine, extraire contributions avec citations exactes, et générer questions de recherche actionnables — tout en conservant traçabilité et seuils de confiance.

**Fonctionnalités clés**

* Ingestion PDF + OCR (si besoin) → parsing titre/abstract.
* Étape 1 : classification domaine (JSON + evidence).
* Étape 2 : extraction contributions (source\_quote obligatoire).
* Étape 3 : génération question de recherche + test minimal + flag human\_review.
* Dashboard d’audit montrant les `source_quote`, confidences et logs.
* Boucle H-in-the-loop pour corrections et fine-tuning incrémental.

**Tech stack recommandé**
Python, LangChain, Hugging Face / OpenAI, FAISS (ou Pinecone) pour vector store, Postgres/Elasticsearch pour métadonnées, Tika / pdfminer / ABBYY, Streamlit ou React pour dashboard, Docker + orchestrateur simple (Celery, Airflow).

**Données nécessaires**
Corpus d’abstracts annotés (100–1000) pour few-shot / évaluation ; échantillon de full-text pour OCR tests.

**Livrables MVP**

* API REST qui prend un PDF/abstract et renvoie les 3 JSON structurés.
* Dashboard montrant 20 documents traités + possibilité de corriger une extraction.
* Suite de tests unitaires (parsing, extraction, classification).

**KPIs / métriques**

* Précision / rappel sur classification domaine (vs étiquettes humaines).
* Exactitude des `source_quote` (proportion de citations correctes).
* % de documents `requires_human_review`.
* Taux d’hallucination détectée (extractions sans source\_quote).

**Risques & mitigations**

* Hallucinations → exiger `source_quote` ; confidence thresholds ; log + human review.
* PDF malformés → fallback OCR + heuristiques robustes.
* Confidentialité → chiffrement en repos/transit et anonymisation.

**Premiers pas**

1. Rassembler 200 abstracts représentatifs par domaine.
2. Écrire 3 prompts (step1/2/3) few-shot et tester localement.
3. Prototyper ingestion + sortie JSON pour 10 documents.

---

## 2) FairCareerAdvisor — reco de carrière anti-biais

**Concept**
Service de recommandation de parcours pro basé uniquement sur compétences et intérêts, avec contraintes de diversification, score traçable et tests counterfactuals automatiques.

**Fonctionnalités clés**

* Normalisation du profil compétences/intérêts.
* Règles de diversification (ex. 2 techniques / 2 soins / 2 créatifs).
* `match_score` explicable via `skill_mapping`.
* Suite d’audit automatique : counterfactual tests (changer genre, âge) et rapport de disparités.
* Interface pour micro-certifs recommandées et chemins d’apprentissage.

**Tech stack recommandé**
Backend Python (FastAPI), embeddings + vector DB, LLM (temperature=0), React front, pipeline d’audit (pytest + scripts), stockage JSON (Mongo/Postgres).

**Données nécessaires**
Dataset métiers (O\*NET/France equivalent) avec compétences requises ; jeux de profils anonymisés pour tests.

**Livrables MVP**

* API qui retourne 6 suggestions JSON conformes au schema `diversity_check`.
* Script d’audit counterfactual produisant rapport CSV.
* Simple UI test permettant d’entrer compétences et d’obtenir suggestions.

**KPIs / métriques**

* Changement des suggestions entre profils counterfactuals (idéal : 0).
* Satisfaction utilisateur (NPS) sur pertinence.
* Taux de respect `diversity_check`.

**Risques & mitigations**

* Biais résiduels → appliquer post-rééquilibrage et filtrage par règles.
* Données métiers insuffisantes → enrichir O\*NET + experts domaine.

**Premiers pas**

1. Extraire dataset métiers + mapping compétences.
2. Écrire prompt role-based `FairCareerAdvisor` et tester 50 profils synthétiques.
3. Développer script counterfactual (swap gender/age) et mesurer variance.

---

## 3) Coach santé mémoire-augmenté (hybride structuré + vectoriel)

**Concept**
Coach personnel qui mémorise préférences/stats (JSON) et extraits de conversation (vector DB) ; utilise ces mémoires pour personnaliser conseils et éviter répétitions.

**Fonctionnalités clés**

* Mémoire structurée pour préférences et conseils donnés.
* Vector store pour snippets sémantiques (recherche top-k).
* Politique d’écriture (confirm before write), TTL pour essais.
* Export / suppression GDPR/HIPAA ready.
* Interface mobile/web et connecteurs optional (Fitbit, Google Fit).

**Tech stack recommandé**
Vector DB (FAISS/Pinecone), Postgres JSONB, OpenAI/HuggingFace, backend (FastAPI), frontend React Native or PWA, chiffrement.

**Données nécessaires**
Scénarios utilisateurs simulés, templates d’interaction, quelques historiques pour RAG.

**Livrables MVP**

* Prototype backend capable d’injecter `profile + relevant_memories` dans prompt et produire JSON de sortie (summary/actions/update\_memory).
* Mini UI pour visualiser mémoire et accepter/rejeter mises à jour.

**KPIs / métriques**

* Taux d’acceptation des actions proposées.
* % de conseils répétés inutilement (devrait décroître).
* Adhérence utilisateur (login cadence).

**Risques & mitigations**

* Vie privée → chiffrement, options suppression.
* Ecriture incorrecte en mémoire → exiger confirmation explicite.

**Premiers pas**

1. Définir schema mémoire minimal et règles d’écriture.
2. Prototyper prompt injectant `RELEVANT_MEMORIES_JSON` et vérifier sorties sur 20 conversations simulées.
3. Intégrer vector store et tester retrievals pertinents.

---

## 4) Classificateur support client robuste (Few-Shot + Ensemble + Follow-up)

**Concept**
Service de classification de messages clients combinant few-shot LLM, ensemble de prompts/modèles, sortie JSON strict, et génération automatique de question de clarification si ambigu.

**Fonctionnalités clés**

* Few-shot prompt + temperature=0 + JSON strict.
* Ensemble voting / calibration (si discordance, basculer à follow\_up).
* Routage automatique (billing/support/account/other) + reason + confidence.
* Dashboard d’erreurs et sample retraining.

**Tech stack recommandé**
LangChain-like orchestration, multiple LLM endpoints (diversifier), Redis queue, Postgres logs, simple front pour supervision.

**Données nécessaires**
Corpus de tickets labellisés (5k idéal, mais prototype avec 500 exemples).

**Livrables MVP**

* Service API classifiant messages et générant JSON.
* Module follow\_up qui produit la question unique si confidence basse ou ambigüité.
* Évaluation automatisée (F1 per class).

**KPIs / métriques**

* Precision / recall / F1 par classe.
* Taux d’escalade humaine (physique measure of ambiguity).
* Temps moyen de traitement par message (observability).

**Risques & mitigations**

* Messages ambigus → follow\_up generation + rule-based priority order.
* Dérive vocabulaire → pipeline d’active learning pour ajouter exemples au few-shot.

**Premiers pas**

1. Rassembler 500 tickets étiquetés, créer few-shot set.
2. Implémenter prompt JSON strict et test sur 100 messages.
3. Ajouter module voting/ensemble pour comparer résultats.

---

## 5) Sandbox AlignedCoT pour vérification multi-chemins

**Concept**
Plateforme expérimentale qui force le modèle à produire **au moins 2 chemins de raisonnement** différents, compare les sorties, détecte divergences et produit une décision vérifiée — utile pour tâches critiques (finance, juridique, médecine).

**Fonctionnalités clés**

* Template AlignedCoT (2+ chemins + comparaison + checklists).
* Moteur d’analyse automatique qui repère inconsistances arithmétiques ou assertions sans source.
* Logging des chaînes de pensée (séparées des réponses finales) avec justificatifs.
* UI montrant chemins côte-à-côte et bouton « expliquer divergence ».

**Tech stack recommandé**
Orchestrateur LLM (parallèle), moteur de validation (python rules), Postgres pour logs, frontend minimal.

**Données nécessaires**
Jeux de cas vérifiables (calculs, Q\&A factuelles) pour évaluer taux de concordance.

**Livrables MVP**

* Endpoint qui prend un item, renvoie chemins A/B, comparaison, réponse finale et verdict.
* Dashboard listant divergences et corrections manuelles.

**KPIs / métriques**

* Taux de concordance entre chemins.
* Taux de corrections manuelles nécessaires.
* Réduction des hallucinations après intégration des checks.

**Risques & mitigations**

* Confidentialité des chaines de pensée → stocker séparément, accès restreint.
* Explosion de tokens → forcer calculs en centimes/chiffres et limiter verbosité.

**Premiers pas**

1. Écrire template AlignedCoT (2 chemins obligatoires + comparaison).
2. Construire 50 cas tests (calculs simples, questions factuelles) et exécuter.
3. Implémenter la règle automatique de détection d’erreur arithmétique (digit-by-digit).

---
